In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask


def find_suim_folders(base_path):
    img_candidates = []
    mask_candidates = []

    for root, dirs, files in os.walk(base_path):
        dset = set(dirs)

        if any(k in root.lower() for k in ["mask", "masks", "seg", "gt"]):
            mask_candidates.append(root)
        if any(k in root.lower() for k in ["image", "images", "img", "rgb"]):
            img_candidates.append(root)

        if "images" in dset and ("masks" in dset or "mask" in dset):
            img_candidates.append(os.path.join(root, "images"))
            if "masks" in dset:
                mask_candidates.append(os.path.join(root, "masks"))
            else:
                mask_candidates.append(os.path.join(root, "mask"))


    def has_imgs(p):
        exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
        return any(len(glob.glob(os.path.join(p, e))) > 0 for e in exts)

    img_candidates = [p for p in img_candidates if os.path.isdir(p) and has_imgs(p)]
    mask_candidates = [p for p in mask_candidates if os.path.isdir(p) and has_imgs(p)]


    if not img_candidates:
        for p in glob.glob(os.path.join(base_path, "**"), recursive=True):
            if os.path.isdir(p) and os.path.basename(p).lower() in ["images", "image", "imgs", "rgb"]:
                if has_imgs(p):
                    img_candidates.append(p)
    if not mask_candidates:
        for p in glob.glob(os.path.join(base_path, "**"), recursive=True):
            if os.path.isdir(p) and os.path.basename(p).lower() in ["masks", "mask", "seg", "gt"]:
                if has_imgs(p):
                    mask_candidates.append(p)


    img_dir = max(img_candidates, key=lambda p: len(glob.glob(os.path.join(p, "*")))) if img_candidates else None
    mask_dir = max(mask_candidates, key=lambda p: len(glob.glob(os.path.join(p, "*")))) if mask_candidates else None

    return img_dir, mask_dir

img_dir, mask_dir = find_suim_folders(path)
print("Found img_dir:", img_dir)
print("Found mask_dir:", mask_dir)

class SUIMDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=(256, 256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size

        exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
        img_paths = []
        for e in exts:
            img_paths.extend(glob.glob(os.path.join(img_dir, e)))
        img_paths = sorted(img_paths)

        mask_paths = []
        mask_map = {}
        for e in exts:
            for p in glob.glob(os.path.join(mask_dir, e)):
                stem = os.path.splitext(os.path.basename(p))[0]
                mask_map[stem] = p

        paired_imgs, paired_masks = [], []
        for ip in img_paths:
            stem = os.path.splitext(os.path.basename(ip))[0]
            if stem in mask_map:
                paired_imgs.append(ip)
                paired_masks.append(mask_map[stem])

        if len(paired_imgs) == 0:
            raise ValueError("No image/mask pairs found. Dataset naming might differ.")

        self.images = paired_imgs
        self.masks = paired_masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self.masks[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")


        image = TF.resize(image, self.size, interpolation=TF.InterpolationMode.BILINEAR)
        mask  = TF.resize(mask,  self.size, interpolation=TF.InterpolationMode.NEAREST)

        image = TF.to_tensor(image)
        mask = torch.from_numpy(np.array(mask)).long()
        mask = remap_mask(mask)

        return image, mask

dataset = SUIMDataset(img_dir, mask_dir, size=(256, 256))


train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Total:", len(dataset), "| Train:", len(train_dataset), "| Val:", len(val_dataset))

# Display samples
def show_samples(ds, n=3):
    fig, axes = plt.subplots(n, 2, figsize=(8, 3*n))
    for i in range(n):
        img, m = ds[i]
        axes[i,0].imshow(img.permute(1,2,0).numpy())
        axes[i,0].set_title("Image")
        axes[i,0].axis("off")

        axes[i,1].imshow(m.numpy(), vmin=0, vmax=7)
        axes[i,1].set_title("Mask")
        axes[i,1].axis("off")
    plt.tight_layout()
    plt.show()


show_samples(dataset, n=3)


In [ ]:
# TO DO

import torch


In [ ]:
# TO DO

!pip -q install segmentation-models-pytorch

import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
    activation=None
).to(device)

print(model.__class__.__name__)



In [ ]:
# TO DO
import torch.nn as nn
from tqdm import tqdm

def pixel_accuracy(logits, masks):

    preds = torch.argmax(logits, dim=1)
    return (preds == masks).float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += pixel_accuracy(logits, masks)

    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks  = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()
        total_acc += pixel_accuracy(logits, masks)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# TO DO
import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 10
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | "
          f"Val Loss: {va_loss:.4f}, Val Acc: {va_acc:.4f}")


plt.figure(figsize=(7,4))
plt.plot(train_losses, marker='o', label="Train Loss")
plt.plot(val_losses, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()


In [ ]:
# TO DO
@torch.no_grad()
def visualize_predictions(model, ds, device, n=3):
    model.eval()
    fig, axes = plt.subplots(n, 3, figsize=(12, 3*n))

    for i in range(n):
        img, mask = ds[i]
        inp = img.unsqueeze(0).to(device)

        logits = model(inp)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu()

        axes[i,0].imshow(img.permute(1,2,0).numpy())
        axes[i,0].set_title("Image")
        axes[i,0].axis("off")

        axes[i,1].imshow(mask.numpy(), vmin=0, vmax=7)
        axes[i,1].set_title("Ground Truth")
        axes[i,1].axis("off")

        axes[i,2].imshow(pred.numpy(), vmin=0, vmax=7)
        axes[i,2].set_title("Prediction")
        axes[i,2].axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(model, dataset, device, n=3)
